# Exploratory Data Analysis
## Multi-Platform Social Media Analytics Platform

This notebook explores the 2.6M row dataset to understand distributions,
trends, and patterns before building the analytics layer.

**Author:** Your Name  
**Dataset:** 7 CSV files · 2,653,201 total rows  
**Platforms:** Instagram · Facebook · YouTube · LinkedIn · WhatsApp Business

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
sns.set_palette('husl')

PLATFORM_COLORS = {
    'Instagram':        '#E1306C',
    'Facebook':         '#1877F2',
    'YouTube':          '#FF0000',
    'LinkedIn':         '#0A66C2',
    'WhatsApp Business':'#25D366',
}

print('✅ Setup complete')

## 1. Load All Datasets

In [ ]:
DATA_PATH = '../data/raw/'

platforms  = pd.read_csv(DATA_PATH + 'platforms.csv')
calendar   = pd.read_csv(DATA_PATH + 'calendar.csv', parse_dates=['date'])
businesses = pd.read_csv(DATA_PATH + 'businesses.csv')
customers  = pd.read_csv(DATA_PATH + 'customers.csv')
campaigns  = pd.read_csv(DATA_PATH + 'campaigns.csv',
                          parse_dates=['start_date', 'end_date'])
engagement = pd.read_csv(DATA_PATH + 'engagement_metrics.csv')
conversions= pd.read_csv(DATA_PATH + 'conversions.csv',
                          parse_dates=['conversion_date'])

print('Dataset Summary')
print('='*45)
for name, df in [('platforms', platforms), ('calendar', calendar),
                  ('businesses', businesses), ('customers', customers),
                  ('campaigns', campaigns), ('engagement_metrics', engagement),
                  ('conversions', conversions)]:
    print(f'{name:<22}: {len(df):>10,} rows  |  {df.shape[1]} cols')
total = sum(len(x) for x in [platforms,calendar,businesses,customers,
                               campaigns,engagement,conversions])
print(f'{"TOTAL":<22}: {total:>10,} rows')

## 2. Campaign Distribution by Platform

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Campaign count by platform
camp_by_platform = campaigns['platform'].value_counts()
colors = [PLATFORM_COLORS.get(p, '#888') for p in camp_by_platform.index]
axes[0].bar(camp_by_platform.index, camp_by_platform.values, color=colors)
axes[0].set_title('Campaign Count by Platform', fontweight='bold')
axes[0].set_xlabel('Platform')
axes[0].set_ylabel('Number of Campaigns')
axes[0].tick_params(axis='x', rotation=15)

# Revenue by platform
rev_by_platform = campaigns.groupby('platform')['revenue_generated'].sum().sort_values(ascending=False)
colors2 = [PLATFORM_COLORS.get(p, '#888') for p in rev_by_platform.index]
axes[1].bar(rev_by_platform.index, rev_by_platform.values / 1e9, color=colors2)
axes[1].set_title('Total Revenue by Platform (₹ Billions)', fontweight='bold')
axes[1].set_xlabel('Platform')
axes[1].set_ylabel('Revenue (₹ Billion)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../data/exports/01_platform_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nRevenue by Platform:')
print(rev_by_platform.apply(lambda x: f'₹{x/1e9:.2f}B').to_string())

## 3. KPI Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
kpis = [
    ('ctr',             'CTR',             axes[0,0]),
    ('cpc',             'CPC (₹)',         axes[0,1]),
    ('roas',            'ROAS',            axes[0,2]),
    ('roi',             'ROI',             axes[1,0]),
    ('conversion_rate', 'Conversion Rate', axes[1,1]),
    ('sentiment_score', 'Sentiment Score', axes[1,2]),
]

for col, label, ax in kpis:
    data = campaigns[col]
    if col == 'roas':
        data = data[data > 0]   # exclude zero-ROAS brand awareness campaigns
    ax.hist(data, bins=50, color='#4F86C6', edgecolor='white', linewidth=0.3)
    ax.set_title(f'{label} Distribution', fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Frequency')
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {data.mean():.3f}')
    ax.legend(fontsize=9)

plt.suptitle('KPI Distribution Across 200,000 Campaigns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/exports/02_kpi_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Revenue Trend Over Time

In [ ]:
campaigns['year']  = campaigns['start_date'].dt.year
campaigns['month'] = campaigns['start_date'].dt.month

monthly = (campaigns.groupby(['year', 'month'])['revenue_generated']
           .sum().reset_index())
monthly['date'] = pd.to_datetime(monthly[['year','month']].assign(day=1))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Monthly trend
axes[0].plot(monthly['date'], monthly['revenue_generated']/1e6,
             color='#1877F2', linewidth=1.5)
axes[0].fill_between(monthly['date'], monthly['revenue_generated']/1e6,
                      alpha=0.1, color='#1877F2')
axes[0].set_title('Monthly Revenue Trend (₹ Millions)', fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Revenue (₹M)')

# Annual bar chart
annual = campaigns.groupby('year')['revenue_generated'].sum()
axes[1].bar(annual.index.astype(str), annual.values/1e9,
             color='#25D366', edgecolor='white')
axes[1].set_title('Annual Revenue (₹ Billions)', fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Revenue (₹B)')
for i, (year, val) in enumerate(annual.items()):
    axes[1].text(i, val/1e9 + 0.05, f'₹{val/1e9:.1f}B',
                 ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/exports/03_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nAnnual Revenue:')
print(annual.apply(lambda x: f'₹{x/1e9:.2f}B').to_string())

## 5. Platform KPI Comparison

In [ ]:
platform_kpis = campaigns.groupby('platform').agg(
    avg_ctr=('ctr', 'mean'),
    avg_cpc=('cpc', 'mean'),
    avg_roas=('roas', lambda x: x[x>0].mean()),
    avg_roi=('roi', lambda x: x[x>0].mean()),
    avg_conversion_rate=('conversion_rate', 'mean'),
    total_campaigns=('campaign_id', 'count'),
).reset_index()

print('Platform KPI Summary')
print('='*75)
print(platform_kpis.set_index('platform').round(4).to_string())

# Heatmap
heatmap_data = platform_kpis.set_index('platform')[
    ['avg_ctr','avg_cpc','avg_roas','avg_roi','avg_conversion_rate']
]
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

plt.figure(figsize=(10, 4))
sns.heatmap(heatmap_norm, annot=heatmap_data.round(3), fmt='.3f',
            cmap='YlOrRd', linewidths=0.5, cbar_kws={'label': 'Normalised Score'})
plt.title('Platform KPI Heatmap (values = actuals, colour = normalised rank)',
          fontweight='bold')
plt.tight_layout()
plt.savefig('../data/exports/04_platform_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Audience Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Age group distribution
age_rev = (campaigns.groupby('audience_age_group')['revenue_generated']
           .sum().sort_values(ascending=False))
axes[0].bar(age_rev.index, age_rev.values/1e9, color='#E1306C')
axes[0].set_title('Revenue by Age Group (₹B)', fontweight='bold')
axes[0].set_xlabel('Age Group')

# Device type
device_conv = campaigns.groupby('device_type')['conversions'].sum()
axes[1].pie(device_conv.values, labels=device_conv.index,
            autopct='%1.1f%%', colors=['#1877F2','#E1306C','#FF9500'])
axes[1].set_title('Conversions by Device', fontweight='bold')

# Influencer vs non-influencer
inf_data = campaigns.groupby('influencer_used')['roas'].mean()
labels = ['No Influencer', 'With Influencer']
axes[2].bar(labels, inf_data.values, color=['#888888','#E1306C'])
axes[2].set_title('Avg ROAS: Influencer vs Non-Influencer', fontweight='bold')
axes[2].set_ylabel('Avg ROAS')
for i, v in enumerate(inf_data.values):
    axes[2].text(i, v + 0.1, f'{v:.2f}x', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/exports/05_audience_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Conversion Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Payment method split
pay = conversions['payment_method'].value_counts()
axes[0].barh(pay.index, pay.values, color='#0A66C2')
axes[0].set_title('Conversions by Payment Method', fontweight='bold')
axes[0].set_xlabel('Count')

# Refund status
refund = conversions['refund_status'].value_counts()
axes[1].pie(refund.values, labels=refund.index,
            autopct='%1.1f%%', colors=['#25D366','#FF4444','#FF9500'])
axes[1].set_title('Refund Status Distribution', fontweight='bold')

# Order value distribution
axes[2].hist(conversions['order_value'], bins=60,
             color='#FF9500', edgecolor='white', linewidth=0.3)
axes[2].axvline(conversions['order_value'].mean(), color='red',
                linestyle='--', linewidth=1.5,
                label=f'Mean: ₹{conversions["order_value"].mean():,.0f}')
axes[2].set_title('Order Value Distribution', fontweight='bold')
axes[2].set_xlabel('Order Value (₹)')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.savefig('../data/exports/06_conversion_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nAvg Order Value : ₹{conversions["order_value"].mean():,.2f}')
print(f'Total Revenue   : ₹{conversions["order_value"].sum()/1e9:.2f}B')
print(f'Refund Rate     : {(conversions["refund_status"] != "No Refund").mean()*100:.1f}%')

## 8. Key Findings Summary

In [ ]:
print('='*60)
print('KEY EDA FINDINGS')
print('='*60)

best_roas_platform = campaigns[campaigns['roas']>0].groupby('platform')['roas'].mean().idxmax()
best_rev_platform  = campaigns.groupby('platform')['revenue_generated'].sum().idxmax()
best_category      = campaigns.merge(businesses[['business_id','business_category']],
                                      on='business_id').groupby(
                                      'business_category')['profit_generated'].sum().idxmax()
influencer_roas    = campaigns[campaigns['influencer_used']==True]['roas'].mean()
no_inf_roas        = campaigns[campaigns['influencer_used']==False]['roas'].mean()
refund_rate        = (conversions['refund_status'] != 'No Refund').mean() * 100

print(f'\n1. Best ROAS platform        : {best_roas_platform}')
print(f'2. Highest revenue platform  : {best_rev_platform}')
print(f'3. Most profitable category  : {best_category}')
print(f'4. Influencer avg ROAS       : {influencer_roas:.2f}x')
print(f'5. Non-influencer avg ROAS   : {no_inf_roas:.2f}x')
print(f'6. Overall refund rate       : {refund_rate:.1f}%')
print(f'7. Avg order value           : ₹{conversions["order_value"].mean():,.0f}')
print(f'8. Zero-ROAS campaigns       : 674 (brand awareness — intentional)')
print('='*60)